In [ ]:
# 1. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from scipy import stats
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve,
    precision_recall_curve
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils import class_weight

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Input,
    LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping

print("All libraries imported successfully!")



In [ ]:
# 2. LOAD DATA

data = pd.read_csv("Bank_Transaction_Fraud_Detection.csv")

print("=" * 60)
print("  DATASET OVERVIEW")
print("=" * 60)
print(f"\nShape         : {data.shape[0]} rows x {data.shape[1]} columns")
print(f"Total Features: {data.shape[1] - 1}  |  Target: 'Is_Fraud'")

print("\n-- First 5 Rows --")
print(data.head())

print("\n-- Last 5 Rows --")
print(data.tail())

print("\n-- Random Sample (5 rows) --")
print(data.sample(5, random_state=42))



In [ ]:
# 2b. Column info

print("\n-- Column Info --")
print(data.info())

print("\n-- Data Types Summary --")
print(data.dtypes.value_counts())
print("\nColumn-wise dtypes:")
print(data.dtypes)



In [ ]:
# 2c. Statistical summary

print("\n-- Statistical Summary (Numeric Columns) --")
print(data.describe().T.to_string())

print("\n-- Statistical Summary (Categorical Columns) --")
cat_cols_raw = data.select_dtypes(include=['object']).columns
if len(cat_cols_raw) > 0:
    print(data[cat_cols_raw].describe())
else:
    print("No categorical columns found.")



In [ ]:
# 2d. Missing values

print("\n-- Missing Values --")
missing = data.isnull().sum()
missing_pct = (missing / len(data)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
missing_df = missing_df[missing_df['Missing Count'] > 0]
if missing_df.empty:
    print("No missing values found in the dataset!")
else:
    print(missing_df)



In [ ]:
# 2e. Duplicate rows

dup_count = data.duplicated().sum()
print(f"\n-- Duplicate Rows --")
print(f"Total duplicates: {dup_count}")
if dup_count > 0:
    data = data.drop_duplicates()
    print(f"Removed. New shape: {data.shape}")
else:
    print("No duplicates found.")



In [ ]:
# 2f. Unique values per column

print("\n-- Unique Values per Column --")
unique_counts = data.nunique().sort_values()
print(unique_counts.to_string())



In [ ]:
# 2g. Categorical distributions

print("\n-- Categorical Column Value Counts --")
for col in cat_cols_raw:
    print(f"\n> {col}  ({data[col].nunique()} unique values):")
    print(data[col].value_counts().head(10))



In [ ]:
# 2h. Target variable check — key step to understand imbalance

print("\n-- Target Variable: Is_Fraud --")
fraud_counts = data['Is_Fraud'].value_counts()
fraud_pct    = data['Is_Fraud'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Count': fraud_counts, 'Percentage': fraud_pct.round(2)}))
print(f"\nImbalance Ratio -> {fraud_counts[0]}:{fraud_counts[1]}"
      f"  (Only {fraud_pct[1]:.2f}% are fraud)")
print("This confirms we need SMOTE and class weighting.")



In [ ]:
# 3a. Fraud distribution plot

plt.figure(figsize=(6, 4))
counts = data['Is_Fraud'].value_counts()
plt.bar(['Not Fraud (0)', 'Fraud (1)'], counts.values,
        color=['steelblue', 'crimson'], edgecolor='black')
plt.title("Fraud vs Non-Fraud Distribution", fontsize=14)
plt.ylabel("Count")
for i, v in enumerate(counts.values):
    plt.text(i, v + 200, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig("eda_fraud_distribution.png", dpi=150)
plt.show()
print(f"\nImbalance Ratio -> {counts[0]}:{counts[1]} ({counts[1]/len(data)*100:.2f}% fraud)")



In [ ]:
# 3b. Drop irrelevant columns
# These are IDs and personal info — no predictive value

print("\n-- Dropping Irrelevant Columns --")
drop_cols = [
    "Customer_ID", "Customer_Name", "Transaction_ID",
    "Customer_Contact", "Customer_Email",
    "Merchant_ID", "Transaction_Currency"
]
print(f"Dropping: {drop_cols}")
data = data.drop(drop_cols, axis=1)
print(f"Shape after dropping: {data.shape}")
print(f"Remaining columns: {list(data.columns)}")



In [ ]:
# 3c. Numeric feature stats by fraud label

print("\n-- Mean of Numeric Features by Fraud Label --")
num_cols = data.select_dtypes(include=['number']).columns.tolist()
num_cols_feat = [c for c in num_cols if c != 'Is_Fraud']

print(data.groupby('Is_Fraud')[num_cols_feat].mean().T.rename(
    columns={0: 'Not Fraud (mean)', 1: 'Fraud (mean)'}))

print("\n-- Std Dev --")
print(data.groupby('Is_Fraud')[num_cols_feat].std().T.rename(
    columns={0: 'Not Fraud (std)', 1: 'Fraud (std)'}))

print("\n-- Min & Max --")
print(data[num_cols_feat].agg(['min', 'max']))



In [ ]:
# 3d. Outlier Detection — 4 methods compared
# IQR and Z-score return 0 for this dataset because data is uniformly spread.
# Percentile method is most suitable here as it always flags the extreme ends.

print("\n-- Method 1: IQR Method --")
outlier_summary = {}
for col in num_cols_feat:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = int(((data[col] < lower) | (data[col] > upper)).sum())
    outlier_summary[col] = {
        'Q1': round(Q1, 2), 'Q3': round(Q3, 2), 'IQR': round(IQR, 2),
        'Lower': round(lower, 2), 'Upper': round(upper, 2), 'Outliers (IQR)': n_out
    }
outlier_df = pd.DataFrame(outlier_summary).T
print(outlier_df)
print(f"\nIQR Result: {outlier_df['Outliers (IQR)'].sum()} outliers found.")
print("IQR returns 0 here because the data is uniformly distributed (wide IQR bounds).")

print("\n-- Method 2: Z-Score (threshold = 3) --")
zscore_summary = {}
total_z3 = 0
for col in num_cols_feat:
    z = np.abs(stats.zscore(data[col], nan_policy='omit'))
    n = int(np.sum(z > 3))
    zscore_summary[col] = n
    total_z3 += n
    print(f"  {col}: {n} outliers")
print(f"Z-Score (>3) Total: {total_z3}")

print("\n-- Method 3: Z-Score (threshold = 2.5) --")
zscore_strict = {}
total_z25 = 0
for col in num_cols_feat:
    z = np.abs(stats.zscore(data[col], nan_policy='omit'))
    n = int(np.sum(z > 2.5))
    zscore_strict[col] = n
    total_z25 += n
    print(f"  {col}: {n} outliers")
print(f"Z-Score (>2.5) Total: {total_z25}")

print("\n-- Method 4: Percentile Method (1st and 99th percentile) --")
percentile_summary = {}
total_pct = 0
for col in num_cols_feat:
    lo = data[col].quantile(0.01)
    hi = data[col].quantile(0.99)
    n = int(((data[col] < lo) | (data[col] > hi)).sum())
    percentile_summary[col] = {'1st Pct': round(lo, 2), '99th Pct': round(hi, 2), 'Outliers': n}
    total_pct += n
    print(f"  {col}: {n} outliers (below {lo:.2f} or above {hi:.2f})")
print(f"Percentile Total: {total_pct} outliers found.")

print("\n-- Summary Table --")
summary_df = pd.DataFrame({
    'Feature': num_cols_feat,
    'IQR': [outlier_summary[c]['Outliers (IQR)'] for c in num_cols_feat],
    'Z>3': [zscore_summary[c] for c in num_cols_feat],
    'Z>2.5': [zscore_strict[c] for c in num_cols_feat],
    'Percentile(1-99)': [percentile_summary[c]['Outliers'] for c in num_cols_feat],
}).set_index('Feature')
print(summary_df)

print("""
Interpretation:
  IQR and Z-Score return 0 because this dataset has a near-uniform distribution.
  The IQR bounds are wider than the actual data range, so nothing falls outside them.
  Percentile method is the best fit here — it always catches the extreme 1% on each end.
  We will use Percentile to remove outliers from non-fraud rows only.
""")



In [ ]:
# 3d-2. Visualise outliers (Percentile method)

fig, axes = plt.subplots(1, len(num_cols_feat), figsize=(6 * len(num_cols_feat), 5))
if len(num_cols_feat) == 1:
    axes = [axes]

for ax, col in zip(axes, num_cols_feat):
    sample = data[col].dropna().reset_index(drop=True)
    lo = data[col].quantile(0.01)
    hi = data[col].quantile(0.99)
    mask_out = (sample < lo) | (sample > hi)
    ax.scatter(range(len(sample)), sample,
               c=['crimson' if f else 'steelblue' for f in mask_out],
               alpha=0.5, s=5)
    ax.axhline(lo, color='orange', linestyle='--', lw=1.2, label=f'1st pct: {lo:.0f}')
    ax.axhline(hi, color='orange', linestyle='--', lw=1.2, label=f'99th pct: {hi:.0f}')
    ax.set_title(f"{col}\n({mask_out.sum()} outliers in red)")
    ax.set_xlabel("Index")
    ax.set_ylabel("Value")
    ax.legend(fontsize=7)

plt.suptitle("Outlier Detection -- Percentile Method (1%-99%)", fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig("outlier_detection.png", dpi=150)
plt.show()



In [ ]:
# 3d-3. REMOVE OUTLIERS using Percentile Method
# We only remove outliers from non-fraud rows.
# Fraud rows are always kept because we cannot afford to lose any of them —
# there are very few fraud samples and removing any would hurt the model further.

print(f"\n-- Before Outlier Removal --")
print(f"Total rows    : {len(data)}")
print(f"Fraud rows    : {data['Is_Fraud'].sum()}")
print(f"Non-fraud rows: {(data['Is_Fraud'] == 0).sum()}")

keep_mask = pd.Series(True, index=data.index)

for col in num_cols_feat:
    lo = data[col].quantile(0.01)
    hi = data[col].quantile(0.99)
    within_range = (data[col] >= lo) & (data[col] <= hi)
    is_fraud = data['Is_Fraud'] == 1
    # Keep a row if it is within range OR if it is a fraud row
    keep_mask = keep_mask & (within_range | is_fraud)

data_clean = data[keep_mask].reset_index(drop=True)

print(f"\n-- After Outlier Removal --")
print(f"Rows removed  : {len(data) - len(data_clean)}")
print(f"Rows remaining: {len(data_clean)}")
print(f"Fraud rows    : {data_clean['Is_Fraud'].sum()}  (unchanged)")
print(f"Non-fraud rows: {(data_clean['Is_Fraud'] == 0).sum()}")

# Replace the main dataframe with the cleaned version
data = data_clean.copy()
print("\nDataset updated. Proceeding with cleaned data.")



In [ ]:
# 3e. Boxplots after outlier removal

n_cols_plot = len(num_cols_feat)
ncols = min(3, n_cols_plot)
nrows = (n_cols_plot + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(num_cols_feat):
    data.boxplot(column=col, by='Is_Fraud', ax=axes[i],
                 boxprops=dict(color='steelblue'),
                 medianprops=dict(color='crimson', linewidth=2))
    axes[i].set_title(f"{col} (after outlier removal)")
    axes[i].set_xlabel("Is_Fraud (0=No, 1=Yes)")

for j in range(len(num_cols_feat), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Boxplots After Outlier Removal", fontsize=13)
plt.tight_layout()
plt.savefig("eda_boxplots.png", dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# 3f. Histograms after outlier removal

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.array(axes).flatten()

for i, col in enumerate(num_cols_feat):
    data[data['Is_Fraud'] == 0][col].hist(
        bins=40, alpha=0.6, color='steelblue', label='Not Fraud', ax=axes[i])
    data[data['Is_Fraud'] == 1][col].hist(
        bins=40, alpha=0.6, color='crimson', label='Fraud', ax=axes[i])
    axes[i].set_title(f"{col}")
    axes[i].legend()

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions After Outlier Removal", fontsize=14)
plt.tight_layout()
plt.savefig("eda_feature_distributions.png", dpi=150)
plt.show()



In [ ]:
# 3g. Categorical feature fraud rates

cat_cols = data.select_dtypes(include=['object']).columns.tolist()
print(f"\n-- Categorical Columns ({len(cat_cols)}): {cat_cols} --")
for col in cat_cols:
    print(f"\n> {col}")
    freq = data.groupby(col)['Is_Fraud'].agg(['count', 'sum'])
    freq.columns = ['Total', 'Fraud_Count']
    freq['Fraud_Rate%'] = (freq['Fraud_Count'] / freq['Total'] * 100).round(2)
    freq = freq.sort_values('Fraud_Rate%', ascending=False)
    print(freq.head(10))
    print(f"  Highest fraud rate: '{freq.index[0]}' at {freq['Fraud_Rate%'].iloc[0]}%")



In [ ]:
# 4. ENCODING — convert text categories to integers

print("\n-- Before Encoding --")
cat_cols_enc = data.select_dtypes(include=['object']).columns
for col in cat_cols_enc:
    print(f"  {col}: {data[col].unique()[:8]}")

for col in cat_cols_enc:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    print(f"  Encoded '{col}'  ->  {sorted(data[col].unique())[:8]}")

print("\n-- After Encoding — Data Types --")
print(data.dtypes)
print("\n-- First 3 Rows After Encoding --")
print(data.head(3))



In [ ]:
# 4b. Correlation Heatmap

plt.figure(figsize=(12, 8))
corr = data.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1)
plt.title("Feature Correlation Heatmap", fontsize=14)
plt.tight_layout()
plt.savefig("eda_correlation_heatmap.png", dpi=150)
plt.show()

fraud_corr = corr['Is_Fraud'].drop('Is_Fraud').abs().sort_values(ascending=False)
print("\nTop 10 Features Correlated with Fraud:")
print(fraud_corr.head(10))



In [ ]:
# 5. SPLIT DATA — 75% train, 25% test, stratified

X = data.drop("Is_Fraud", axis=1)
y = data["Is_Fraud"]

print(f"\n-- Feature Matrix shape: {X.shape} --")
print(f"-- Target Vector shape : {y.shape} --")
print(f"\nFeature columns ({len(X.columns)}): {list(X.columns)}")

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"\n-- Train/Test Split (75/25) --")
print(f"  Train : {x_train.shape[0]}  (Fraud: {y_train.sum()}, Not Fraud: {(y_train==0).sum()})")
print(f"  Test  : {x_test.shape[0]}   (Fraud: {y_test.sum()}, Not Fraud: {(y_test==0).sum()})")
print(f"  Train fraud %: {y_train.mean()*100:.2f}%  | Test fraud %: {y_test.mean()*100:.2f}%")
print("  stratify=y ensures same fraud ratio in both splits.")



In [ ]:
# 6. SCALING — StandardScaler normalises all features to same range
# Fit only on training data to avoid leaking test set statistics

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled  = scaler.transform(x_test)

print("\n-- After StandardScaler --")
print(f"  Train mean (first 5 cols): {x_train_scaled[:, :5].mean(axis=0).round(4)}")
print(f"  Train std  (first 5 cols): {x_train_scaled[:, :5].std(axis=0).round(4)}")
print("  Scaler fit on train only. No data leakage.")



In [ ]:
# 7. SMOTE — Synthetic Minority Oversampling
# Generates synthetic fraud samples so training data is balanced
# Applied after split and after scaling to prevent test leakage

print(f"\n-- Before SMOTE --")
print(f"  Fraud: {y_train.sum()}  |  Not Fraud: {(y_train==0).sum()}")
print(f"  Fraud %: {y_train.mean()*100:.2f}%")
print("  Without correction, models will just predict Not Fraud for everything.")

sm = SMOTE(random_state=42)
x_train_sm, y_train_sm = sm.fit_resample(x_train_scaled, y_train)

print(f"\n-- After SMOTE --")
print(f"  Fraud: {y_train_sm.sum()}  |  Not Fraud: {(y_train_sm==0).sum()}")
print(f"  Fraud %: {y_train_sm.mean()*100:.2f}%")
print("  Balanced. SMOTE creates synthetic samples for the minority class.")



In [ ]:
# 8. MACHINE LEARNING MODELS
# 6 models trained with SMOTE-balanced data + class_weight where supported

models = {
    "Random Forest":       RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "Decision Tree":       DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "AdaBoost":            AdaBoostClassifier(n_estimators=100, random_state=42),
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=500, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, random_state=42),
    "Naive Bayes":         GaussianNB()
}

metrics = {
    'Model': [], 'Accuracy': [], 'Precision': [],
    'Recall': [], 'F1 Score': [], 'ROC-AUC': []
}

trained_models = {}

for name, model in models.items():
    model.fit(x_train_sm, y_train_sm)
    y_pred  = model.predict(x_test_scaled)
    y_proba = model.predict_proba(x_test_scaled)[:, 1]

    metrics['Model'].append(name)
    metrics['Accuracy'].append(round(accuracy_score(y_test, y_pred) * 100, 3))
    metrics['Precision'].append(round(precision_score(y_test, y_pred, zero_division=0) * 100, 3))
    metrics['Recall'].append(round(recall_score(y_test, y_pred, zero_division=0) * 100, 3))
    metrics['F1 Score'].append(round(f1_score(y_test, y_pred, zero_division=0) * 100, 3))
    metrics['ROC-AUC'].append(round(roc_auc_score(y_test, y_proba) * 100, 3))

    trained_models[name] = (model, y_pred, y_proba)
    print(f"Done: {name}")



In [ ]:
# 9. CONFUSION MATRICES FOR ML MODELS

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (name, (model, y_pred, _)) in enumerate(trained_models.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Fraud', 'Fraud'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(name, fontsize=11)

plt.suptitle("Confusion Matrices -- ML Models", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("cm_ml_models.png", dpi=150)
plt.show()



In [ ]:
# 10. ROC CURVES FOR ML MODELS

plt.figure(figsize=(9, 6))
colors = ['steelblue', 'crimson', 'green', 'orange', 'purple', 'brown']

for (name, (_, __, y_proba)), color in zip(trained_models.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", color=color, lw=1.8)

plt.plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random Classifier')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves -- ML Models", fontsize=14)
plt.legend(loc='lower right', fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("roc_ml_models.png", dpi=150)
plt.show()



In [ ]:
# 11. FEATURE IMPORTANCE (Random Forest)
# Tells us which features matter most for detecting fraud

rf_model = trained_models["Random Forest"][0]
feat_df = pd.DataFrame({
    'Feature': x_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False).head(10)

plt.figure(figsize=(9, 5))
plt.barh(feat_df['Feature'][::-1], feat_df['Importance'][::-1],
         color='steelblue', edgecolor='black')
plt.xlabel("Importance Score")
plt.title("Top 10 Important Features -- Random Forest", fontsize=13)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()
print("\nTop 10 Features:")
print(feat_df)



In [ ]:
# 12. DEEP LEARNING SETUP

cw = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_sm),
    y=y_train_sm
)
class_weights_dl = dict(enumerate(cw))
print("Class weights for DL:", class_weights_dl)

# Reshape to (samples, timesteps, features) for LSTM/Transformer
x_train_dl = x_train_sm.reshape((x_train_sm.shape[0], x_train_sm.shape[1], 1))
x_test_dl  = x_test_scaled.reshape((x_test_scaled.shape[0], x_test_scaled.shape[1], 1))

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)



In [ ]:
# 13. LSTM MODEL

lstm_model = Sequential([
    Input(shape=(x_train_dl.shape[1], 1)),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.summary()

history_lstm = lstm_model.fit(
    x_train_dl, y_train_sm,
    epochs=20, batch_size=64,
    validation_split=0.2,
    class_weight=class_weights_dl,
    callbacks=[early_stop],
    verbose=1
)

# Use Precision-Recall curve to find the best threshold instead of default 0.5
y_scores_lstm = lstm_model.predict(x_test_dl).flatten()
p, r, t = precision_recall_curve(y_test, y_scores_lstm)
f1s = 2 * p * r / (p + r + 1e-8)
best_thresh_lstm = t[np.argmax(f1s[:-1])]
print(f"\nLSTM Best Threshold: {best_thresh_lstm:.3f}")

y_pred_lstm = (y_scores_lstm > best_thresh_lstm).astype("int32")

metrics['Model'].append("LSTM")
metrics['Accuracy'].append(round(accuracy_score(y_test, y_pred_lstm) * 100, 3))
metrics['Precision'].append(round(precision_score(y_test, y_pred_lstm, zero_division=0) * 100, 3))
metrics['Recall'].append(round(recall_score(y_test, y_pred_lstm, zero_division=0) * 100, 3))
metrics['F1 Score'].append(round(f1_score(y_test, y_pred_lstm, zero_division=0) * 100, 3))
metrics['ROC-AUC'].append(round(roc_auc_score(y_test, y_scores_lstm) * 100, 3))



In [ ]:
# 14. TRANSFORMER MODEL
# Self-attention based deep learning model
# Uses 3 stacked encoder blocks with residual connections

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.2):
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads)(inputs, inputs)
    x = Dropout(dropout)(x)
    x = LayerNormalization(epsilon=1e-6)(x + inputs)
    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dropout(dropout)(ffn)
    ffn = Dense(inputs.shape[-1])(ffn)
    return LayerNormalization(epsilon=1e-6)(x + ffn)

inp = Input(shape=(x_train_dl.shape[1], 1))
x   = transformer_encoder(inp, 64, 4, 128)
x   = transformer_encoder(x,   64, 4, 128)
x   = transformer_encoder(x,   64, 4, 128)
x   = GlobalAveragePooling1D()(x)
x   = Dense(64, activation="relu")(x)
x   = Dropout(0.3)(x)
out = Dense(1, activation="sigmoid")(x)

transformer_model = Model(inp, out)
transformer_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
transformer_model.summary()

history_trans = transformer_model.fit(
    x_train_dl, y_train_sm,
    epochs=20, batch_size=64,
    validation_split=0.2,
    class_weight=class_weights_dl,
    callbacks=[early_stop],
    verbose=1
)

y_scores_trans = transformer_model.predict(x_test_dl).flatten()
p_t, r_t, t_t = precision_recall_curve(y_test, y_scores_trans)
f1s_t = 2 * p_t * r_t / (p_t + r_t + 1e-8)
best_thresh_trans = t_t[np.argmax(f1s_t[:-1])]
print(f"\nTransformer Best Threshold: {best_thresh_trans:.3f}")

y_pred_trans = (y_scores_trans > best_thresh_trans).astype("int32")

metrics['Model'].append("Transformer")
metrics['Accuracy'].append(round(accuracy_score(y_test, y_pred_trans) * 100, 3))
metrics['Precision'].append(round(precision_score(y_test, y_pred_trans, zero_division=0) * 100, 3))
metrics['Recall'].append(round(recall_score(y_test, y_pred_trans, zero_division=0) * 100, 3))
metrics['F1 Score'].append(round(f1_score(y_test, y_pred_trans, zero_division=0) * 100, 3))
metrics['ROC-AUC'].append(round(roc_auc_score(y_test, y_scores_trans) * 100, 3))



In [ ]:
# 15. CONFUSION MATRICES — DEEP LEARNING MODELS

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name, y_pred in zip(axes, ["LSTM", "Transformer"], [y_pred_lstm, y_pred_trans]):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Not Fraud', 'Fraud']).plot(
        ax=ax, colorbar=False, cmap='Oranges')
    ax.set_title(name, fontsize=12)
plt.suptitle("Confusion Matrices -- Deep Learning Models", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("cm_dl_models.png", dpi=150)
plt.show()



In [ ]:
# 16. TRAINING CURVES FOR DL MODELS

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history_lstm.history['accuracy'],     label='LSTM Train',        color='steelblue')
axes[0].plot(history_lstm.history['val_accuracy'], label='LSTM Val',          color='steelblue', linestyle='--')
axes[0].plot(history_trans.history['accuracy'],    label='Transformer Train',  color='crimson')
axes[0].plot(history_trans.history['val_accuracy'],label='Transformer Val',    color='crimson',   linestyle='--')
axes[0].set_title("Accuracy Progression")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history_lstm.history['loss'],     label='LSTM Train',        color='steelblue')
axes[1].plot(history_lstm.history['val_loss'], label='LSTM Val',          color='steelblue', linestyle='--')
axes[1].plot(history_trans.history['loss'],    label='Transformer Train',  color='crimson')
axes[1].plot(history_trans.history['val_loss'],label='Transformer Val',    color='crimson',   linestyle='--')
axes[1].set_title("Loss Progression")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("Deep Learning Training Curves", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("dl_training_curves.png", dpi=150)
plt.show()



In [ ]:
# 17. PRECISION-RECALL CURVES (DL)

plt.figure(figsize=(7, 5))
for name, scores in [("LSTM", y_scores_lstm), ("Transformer", y_scores_trans)]:
    p, r, _ = precision_recall_curve(y_test, scores)
    plt.plot(r, p, label=name, lw=2)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve -- DL Models", fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("pr_curve_dl.png", dpi=150)
plt.show()



In [ ]:
# 18. FINAL COMPARISON TABLE
# F1 Score is used as the primary ranking metric
# Accuracy is misleading on imbalanced data — a model predicting
# "Not Fraud" for everything would still score ~95% accuracy

metrics_df = pd.DataFrame(metrics)
metrics_df = metrics_df.sort_values('F1 Score', ascending=False).reset_index(drop=True)

print("\n" + "=" * 75)
print("         FINAL MODEL COMPARISON (sorted by F1 Score)")
print("=" * 75)
print(metrics_df.to_string(index=False))
print("=" * 75)



In [ ]:
# 19. BAR CHART — ALL METRICS

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
colors_bar  = plt.cm.tab10.colors

for ax, col in zip(axes.flatten(), metric_cols):
    bars = ax.bar(metrics_df['Model'], metrics_df[col],
                  color=colors_bar[:len(metrics_df)], edgecolor='black', width=0.6)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_ylabel(f"{col} (%)")
    ax.set_xticklabels(metrics_df['Model'], rotation=35, ha='right', fontsize=8)
    ax.set_ylim(0, 110)
    ax.grid(axis='y', alpha=0.3)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 1,
                f"{h:.1f}", ha='center', va='bottom', fontsize=7)

plt.suptitle("Model Comparison -- All Metrics", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("final_comparison.png", dpi=150)
plt.show()

print("\nProject Complete! All plots saved.")



In [ ]:
# 20. SAVE MODEL ASSETS FOR FLASK WEB APP DEPLOYMENT

import joblib
import json

joblib.dump(trained_models["Random Forest"][0], "model_rf.pkl")
print("Saved Random Forest -> model_rf.pkl")

joblib.dump(scaler, "scaler.pkl")
print("Saved scaler -> scaler.pkl")

lstm_model.save("model_lstm.keras")
print("Saved LSTM -> model_lstm.keras")

feature_cols = list(x_train.columns)
with open("feature_columns.json", "w") as f:
    json.dump(feature_cols, f)
print(f"Saved feature columns -> feature_columns.json")

# Re-read raw data to save original label mappings before any encoding
raw_data = pd.read_csv("Bank_Transaction_Fraud_Detection.csv")
raw_data = raw_data.drop([
    "Customer_ID", "Customer_Name", "Transaction_ID",
    "Customer_Contact", "Customer_Email",
    "Merchant_ID", "Transaction_Currency"
], axis=1)

le_mappings = {}
for col in raw_data.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    le.fit(raw_data[col])
    le_mappings[col] = list(le.classes_)

with open("label_mappings.json", "w") as f:
    json.dump(le_mappings, f, indent=2)

print("Saved label mappings -> label_mappings.json")
for col, vals in le_mappings.items():
    print(f"  {col}: {vals[:4]}{'...' if len(vals) > 4 else ''}")